# Overall Scoring Correlation Analysis

This notebook compares four scoring cases and reports correlation of image difficulty score with:
- missed detections rate
- false positive rate

Inputs are fixed to one scoring JSON per case (seed 123 by default).

In [3]:
from pathlib import Path
import json
import numpy as np

# One input file per requested case (edit paths if needed)
case_to_score_path = {
    "MinneApple | with traditional aug": Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/minneapple/scoring/seed_123/score_results.json"),
    "MinneApple | without traditional aug": Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_123/score_results.json"),
    "GWHD | with traditional aug": Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/gwhd/scoring/seed_123/score_results.json"),
    "GWHD | without traditional aug": Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/gwhd/scoring/seed_123/score_results.json"),
}


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    if np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def rankdata(arr: np.ndarray) -> np.ndarray:
    order = np.argsort(arr)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(arr), dtype=float)
    _, inv, counts = np.unique(arr, return_inverse=True, return_counts=True)
    for i, c in enumerate(counts):
        if c > 1:
            idx = np.where(inv == i)[0]
            ranks[idx] = ranks[idx].mean()
    return ranks + 1.0


rows = []
for case_name, score_path in case_to_score_path.items():
    if not score_path.exists():
        rows.append(
            {
                "case": case_name,
                "n_images": 0,
                "selected_object_weight": np.nan,
                "selected_false_positive_weight": np.nan,
                "pearson(score, miss_rate)": np.nan,
                "spearman(score, miss_rate)": np.nan,
                "pearson(score, fp_rate)": np.nan,
                "spearman(score, fp_rate)": np.nan,
            }
        )
        continue

    with open(score_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_difficulties = data.get("image_difficulties", [])
    selected_w1 = float(data.get("selected_object_weight", np.nan))
    selected_w2 = float(data.get("selected_false_positive_weight", np.nan))

    if len(image_difficulties) == 0:
        rows.append(
            {
                "case": case_name,
                "n_images": 0,
                "selected_object_weight": selected_w1,
                "selected_false_positive_weight": selected_w2,
                "pearson(score, miss_rate)": np.nan,
                "spearman(score, miss_rate)": np.nan,
                "pearson(score, fp_rate)": np.nan,
                "spearman(score, fp_rate)": np.nan,
            }
        )
        continue

    score = np.asarray([float(img.get("difficulty_score", np.nan)) for img in image_difficulties], dtype=float)
    miss = np.asarray([float(img.get("missed_detections_rate", np.nan)) for img in image_difficulties], dtype=float)
    fp = np.asarray([float(img.get("false_positive_rate", np.nan)) for img in image_difficulties], dtype=float)

    valid_miss = ~np.isnan(score) & ~np.isnan(miss)
    valid_fp = ~np.isnan(score) & ~np.isnan(fp)

    pearson_miss = safe_corr(score[valid_miss], miss[valid_miss])
    spearman_miss = safe_corr(rankdata(score[valid_miss]), rankdata(miss[valid_miss])) if np.sum(valid_miss) >= 2 else np.nan

    pearson_fp = safe_corr(score[valid_fp], fp[valid_fp])
    spearman_fp = safe_corr(rankdata(score[valid_fp]), rankdata(fp[valid_fp])) if np.sum(valid_fp) >= 2 else np.nan

    rows.append(
        {
            "case": case_name,
            "n_images": int(len(image_difficulties)),
            "selected_object_weight": selected_w1,
            "selected_false_positive_weight": selected_w2,
            "pearson(score, miss_rate)": pearson_miss,
            "spearman(score, miss_rate)": spearman_miss,
            "pearson(score, fp_rate)": pearson_fp,
            "spearman(score, fp_rate)": spearman_fp,
        }
    )

# Output as a table
try:
    import pandas as pd
    table = pd.DataFrame(rows)
    for col in [
        "selected_object_weight",
        "selected_false_positive_weight",
        "pearson(score, miss_rate)",
        "spearman(score, miss_rate)",
        "pearson(score, fp_rate)",
        "spearman(score, fp_rate)",
    ]:
        table[col] = table[col].map(lambda x: f"{x:.4f}" if not np.isnan(x) else "nan")
    display(table)
except Exception:
    header = [
        "case",
        "n_images",
        "selected_object_weight",
        "selected_false_positive_weight",
        "pearson(score, miss_rate)",
        "spearman(score, miss_rate)",
        "pearson(score, fp_rate)",
        "spearman(score, fp_rate)",
    ]
    print(" | ".join(header))
    print("-" * 160)
    for r in rows:
        print(
            f"{r['case']} | {r['n_images']} | {r['selected_object_weight']:.4f} | {r['selected_false_positive_weight']:.4f} | "
            f"{r['pearson(score, miss_rate)']:.4f} | {r['spearman(score, miss_rate)']:.4f} | "
            f"{r['pearson(score, fp_rate)']:.4f} | {r['spearman(score, fp_rate)']:.4f}"
        )

,case,n_images,selected_object_weight,selected_false_positive_weight,"pearson(score, miss_rate)","spearman(score, miss_rate)","pearson(score, fp_rate)","spearman(score, fp_rate)"
0,MinneApple | with traditional aug,536,0.9000,0.3450,0.6978,0.6810,0.6883,0.6498
1,MinneApple | without traditional aug,536,0.9000,0.9406,0.6788,0.6675,0.6983,0.6463
2,GWHD | with traditional aug,2700,0.9000,0.7506,0.8158,0.7519,0.8180,0.6652
3,GWHD | without traditional aug,2700,0.9000,0.6852,0.7180,0.6689,0.7157,0.6194
